# Getting Started with PropFlow: A Beginner's Tutorial

This notebook introduces you to **PropFlow**, a Python toolkit for experimenting with belief propagation (BP) and distributed constraint optimization (DCOP) algorithms on factor graphs.

## What You'll Learn

1. What are factor graphs and DCOPs
2. Creating a simple factor graph from scratch
3. Running your first belief propagation algorithm
4. Understanding the results
5. Using FGBuilder for easier graph creation
6. Different BP variants (computators)

## What is a Factor Graph?

A **factor graph** is a bipartite graph used to represent factorizations of functions. In the context of DCOPs:

- **Variable nodes** represent decision variables (each has a domain of possible values)
- **Factor nodes** represent constraints or cost functions between variables
- **Edges** connect factors to the variables they constrain

The goal is to find an assignment of values to variables that minimizes the total cost.

### Example: Meeting Scheduling

Imagine scheduling a meeting between Alice, Bob, and Charlie. Each person can meet at time slots 0, 1, or 2.
- Some time slots cost more for some people (they're busy)
- If Alice and Bob choose different times, there's a coordination cost
- Same for Bob and Charlie

This is a DCOP! Let's model it with PropFlow.

In [1]:
# imports
import numpy as np
import matplotlib.pyplot as plt
from propflow import (
    FactorGraph,
    VariableAgent,
    FactorAgent,
    BPEngine,
    MinSumComputator,
    MaxSumComputator,
    SumProductComputator,
)

print("PropFlow imported successfully!")

PropFlow imported successfully!


## Step 1: Create Variables

Each person is a variable with domain [0, 1, 2] representing time slots.

In [2]:
# create three variable agents - one for each person
alice = VariableAgent(name="Alice", domain=3)  # domain=3 means values {0, 1, 2}
bob = VariableAgent(name="Bob", domain=3)
charlie = VariableAgent(name="Charlie", domain=3)

print(f"Created variables: {alice.name}, {bob.name}, {charlie.name}")
print(f"Each can take values: {list(range(alice.domain))}")

Created variables: Alice, Bob, Charlie
Each can take values: [0, 1, 2]


## Step 2: Create Cost Tables

Cost tables define the costs for different value combinations. Let's create:
1. A constraint between Alice and Bob
2. A constraint between Bob and Charlie

The cost table is a numpy array where `cost_table[i, j]` is the cost when first variable takes value `i` and second takes value `j`.

In [3]:
# cost between Alice and Bob
# we want them to meet at the same time, so diagonal is cheap, off-diagonal is expensive
alice_bob_costs = np.array([
    [0, 10, 10],   # if Alice picks 0: cost 0 if Bob picks 0, cost 10 otherwise
    [10, 0, 10],   # if Alice picks 1: cost 0 if Bob picks 1, cost 10 otherwise
    [10, 10, 0]    # if Alice picks 2: cost 0 if Bob picks 2, cost 10 otherwise
])

# cost between Bob and Charlie - similar coordination
bob_charlie_costs = np.array([
    [0, 15, 15],
    [15, 0, 15],
    [15, 15, 0]
])

print("Alice-Bob cost table:")
print(alice_bob_costs)
print("\nBob-Charlie cost table:")
print(bob_charlie_costs)

Alice-Bob cost table:
[[ 0 10 10]
 [10  0 10]
 [10 10  0]]

Bob-Charlie cost table:
[[ 0 15 15]
 [15  0 15]
 [15 15  0]]


## Step 3: Create Factor Agents

Factors encode the constraints. We pass the cost table directly.

In [5]:
# create factor for Alice-Bob constraint
f_alice_bob = FactorAgent.create_from_cost_table(
    name="f_AB",  # matches the variable domains
    cost_table=alice_bob_costs
)

# create factor for Bob-Charlie constraint
f_bob_charlie = FactorAgent.create_from_cost_table(
    name="f_BC",  # matches the variable domains
    cost_table=bob_charlie_costs
)

print(f"Created factors: {f_alice_bob.name}, {f_bob_charlie.name}")

Created factors: f_AB, f_BC


## Step 4: Build the Factor Graph

Now we connect everything together. The `edges` dictionary maps each factor to its connected variables.

In [8]:
# define the edges: which variables connect to which factors
edges = {
    f_alice_bob: [alice, bob],      # this factor constrains Alice and Bob
    f_bob_charlie: [bob, charlie]   # this factor constrains Bob and Charlie
}

# create the factor graph
fg = FactorGraph(
    variable_li=[alice, bob, charlie],
    factor_li=[f_alice_bob, f_bob_charlie],
    edges=edges
)

print(f"Factor graph created!")
print(f"  Variables: {len(fg.variables)}")
print(f"  Factors: {len(fg.factors)}")
print(f"  Edges: {fg.G.number_of_edges()}")

Factor graph created!
  Variables: 3
  Factors: 2
  Edges: 4


## Step 5: Run Belief Propagation

Now let's solve the problem using the **Min-Sum** algorithm, a variant of belief propagation for finding minimum cost assignments.

In [ ]:
# create the BP engine with Min-Sum computator
engine = BPEngine(
    factor_graph=fg,
    computator=MinSumComputator()
)

# run for 20 iterations
engine.run(max_iter=20)
print("\nBelief Propagation completed!")
print(engine.convergence_monitor.get_convergence_summary())
print(f"Iterations run: {engine.step}")


Belief Propagation completed!
{'total_iterations': 5, 'converged': False, 'final_max_belief_change': np.float64(0.0), 'history': [{'iteration': 2, 'max_belief_change': np.float64(0.0), 'belief_converged': np.True_, 'assignment_converged': True}, {'iteration': 3, 'max_belief_change': np.float64(0.0), 'belief_converged': np.True_, 'assignment_converged': True}, {'iteration': 4, 'max_belief_change': np.float64(0.0), 'belief_converged': np.True_, 'assignment_converged': True}, {'iteration': 5, 'max_belief_change': np.float64(0.0), 'belief_converged': np.True_, 'assignment_converged': True}]}
Iterations run: <bound method BPEngine.step of <propflow.bp.engine_base.BPEngine object at 0x152b5e5d0>>


## Step 6: Examine the Results

In [ ]:
# get the final assignments
assignments = engine.assignments
print("Final assignments:")
for var_name, value in assignments.items():
    print(f"  {var_name}: time slot {value}")

# calculate the total cost
final_cost = engine.calculate_global_cost()
print(f"\nTotal cost: {final_cost}")

# get belief distributions (how confident the algorithm is about each value)
beliefs = engine.get_beliefs()
print("\nBelief distributions:")
for var_name, belief in beliefs.items():
    print(f"  {var_name}: {belief}")

### Understanding the Results

- **Assignments**: The best value for each variable (minimizes total cost)
- **Total cost**: The sum of all constraint costs given the assignments
- **Beliefs**: The algorithm's confidence distribution over each variable's domain

Since we designed the cost tables to prefer everyone meeting at the same time, BP should find that all three should pick the same time slot!

In [ ]:
# plot the cost convergence over iterations
plt.figure(figsize=(10, 5))
plt.plot(engine.history.costs, marker='o', linewidth=2)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Global Cost', fontsize=12)
plt.title('Cost Convergence During Belief Propagation', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Starting cost: {engine.history.costs[0]:.2f}")
print(f"Final cost: {engine.history.costs[-1]:.2f}")
print(f"Improvement: {engine.history.costs[0] - engine.history.costs[-1]:.2f}")

## Using FGBuilder for Easier Graph Creation

For larger problems, creating graphs manually is tedious. PropFlow provides **FGBuilder** to generate graphs automatically.

In [ ]:
from propflow import FGBuilder, CTFactories

# create a cycle graph with 6 variables
cycle_graph = FGBuilder.build_cycle_graph(
    num_vars=6,
    domain_size=4,
    ct_factory=CTFactories.RANDOM_INT.fn,  # random integer cost tables
    ct_params={'low': 0, 'high': 20},
    density=1.0  # not used for cycle graphs
)

print(f"Created cycle graph with {len(cycle_graph.variables)} variables")
print(f"Each variable has domain size {cycle_graph.variables[0].domain}")
print(f"Number of factors: {len(cycle_graph.factors)}")

In [ ]:
# run BP on the cycle graph
engine_cycle = BPEngine(
    factor_graph=cycle_graph,
    computator=MinSumComputator()
)
engine_cycle.run(max_iter=50)

print(f"\nCycle graph solved!")
print(f"Converged: {engine_cycle.converged}")
print(f"Final cost: {engine_cycle.calculate_global_cost():.2f}")
print(f"Assignments: {engine_cycle.assignments}")

In [ ]:
# create a random graph with custom density
random_graph = FGBuilder.build_random_graph(
    num_vars=10,
    domain_size=5,
    ct_factory=CTFactories.RANDOM_INT.fn,
    ct_params={'low': 10, 'high': 50},
    density=0.3  # 30% chance of edge between any variable pair
)

print(f"\nCreated random graph with {len(random_graph.variables)} variables")
print(f"Number of factors: {len(random_graph.factors)}")

# solve it
engine_random = BPEngine(
    factor_graph=random_graph,
    computator=MinSumComputator()
)
engine_random.run(max_iter=100)

print(f"\nRandom graph solved!")
print(f"Final cost: {engine_random.calculate_global_cost():.2f}")

## Different BP Variants: Computators

PropFlow supports different message-passing algebras through **computators**:

- **MinSumComputator**: For minimizing costs (optimization)
- **MaxSumComputator**: For maximizing rewards (optimization)
- **SumProductComputator**: For marginal probability inference
- **MaxProductComputator**: For MAP inference

Let's compare Min-Sum and Max-Sum on the same problem.

In [ ]:
# create a test graph
test_graph = FGBuilder.build_cycle_graph(
    num_vars=5,
    domain_size=3,
    ct_factory=CTFactories.RANDOM_INT.fn,
    ct_params={'low': 0, 'high': 30},
)

# run with Min-Sum (minimize)
engine_min = BPEngine(factor_graph=test_graph, computator=MinSumComputator())
engine_min.run(max_iter=50)

# run with Max-Sum (maximize)
engine_max = BPEngine(factor_graph=test_graph, computator=MaxSumComputator())
engine_max.run(max_iter=50)

print("Min-Sum results (minimize cost):")
print(f"  Final cost: {engine_min.calculate_global_cost():.2f}")
print(f"  Assignments: {engine_min.assignments}")

print("\nMax-Sum results (maximize reward):")
print(f"  Final reward: {engine_max.calculate_global_cost():.2f}")
print(f"  Assignments: {engine_max.assignments}")

## Comparing Convergence

Let's visualize how Min-Sum and Max-Sum converge differently.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(engine_min.history.costs, marker='o', linewidth=2, color='blue', label='Min-Sum')
plt.xlabel('Iteration', fontsize=11)
plt.ylabel('Cost', fontsize=11)
plt.title('Min-Sum: Minimizing Cost', fontsize=13)
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(engine_max.history.costs, marker='s', linewidth=2, color='red', label='Max-Sum')
plt.xlabel('Iteration', fontsize=11)
plt.ylabel('Reward', fontsize=11)
plt.title('Max-Sum: Maximizing Reward', fontsize=13)
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## Summary

In this tutorial, you learned:

1. ✅ **Factor graphs** represent DCOPs with variables, factors, and edges
2. ✅ **Creating graphs manually** using VariableAgent, FactorAgent, and FactorGraph
3. ✅ **Running BP** with the BPEngine class
4. ✅ **Interpreting results** (assignments, costs, beliefs)
5. ✅ **Using FGBuilder** for automatic graph generation
6. ✅ **Different computators** for different BP variants

## Next Steps

- **Tutorial 02**: Learn about engine policies (damping, splitting, etc.) and the Simulator
- **Tutorial 03**: Solve real-world problems like graph coloring and scheduling
- Check the [PropFlow documentation](https://ormullerhahitti.github.io/Belief-Propagation-Simulator/) for more advanced features